# 2 · The recommenders, side by side
Same target user, five methods. Interests are the primary signal; the last cell adds the
second-stage re-rank (mutual friends, places in common, recency).

In [1]:
import pandas as pd
from recsys import db
from recsys.models import Recommender, DEFAULT_WEIGHTS
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)

conn = db.connect(db.DEFAULT_DB) if db.DEFAULT_DB.exists() else db.connect(db.build_db())
profiles, conns, signals = db.load_profiles(conn), db.load_connections(conn), db.load_signals(conn)
target = 9
cands = db.candidate_ids(conn, target)
print(profiles.set_index("user_id").loc[target, "name"], "likes:", profiles.set_index("user_id").loc[target, "interests"])

Sana Sharma likes: ['astronomy and stargazing', 'hanging out with friends', 'marvel movie nights', 'scrolling social media', 'star trek marathons', 'watching movies', 'watching sci-fi movies']


**Jaccard** (baseline): counts exact overlap. Sharing ubiquitous "listening to music" counts as much as a rare hobby.

In [2]:
Recommender("jaccard").fit(profiles).recommend(target, 5, cands)

,user_id,name,score,shared_interests
0,4,Arjun Bhat,0.5,"[marvel movie nights, scrolling social media, star trek marathons, watching sci-fi mov..."
1,16,Meera Rao,0.5,"[hanging out with friends, marvel movie nights, star trek marathons, watching movies, ..."
2,17,Varun Menon,0.5,"[astronomy and stargazing, hanging out with friends, scrolling social media, watching ..."
3,20,Ananya Reddy,0.5,"[astronomy and stargazing, marvel movie nights, scrolling social media, watching movie..."
4,23,Kabir Hegde,0.5,"[hanging out with friends, scrolling social media, star trek marathons, watching sci-f..."


**TF-IDF + cosine**: rare shared interests weigh more. It is still exact-match only.

In [3]:
Recommender("tfidf").fit(profiles).recommend(target, 5, cands)

,user_id,name,score,shared_interests
0,4,Arjun Bhat,0.7359,"[marvel movie nights, scrolling social media, star trek marathons, watching sci-fi mov..."
1,16,Meera Rao,0.6820,"[hanging out with friends, marvel movie nights, star trek marathons, watching movies, ..."
2,17,Varun Menon,0.6707,"[astronomy and stargazing, hanging out with friends, scrolling social media, watching ..."
3,20,Ananya Reddy,0.6494,"[astronomy and stargazing, marvel movie nights, scrolling social media, watching movie..."
4,1,Isha Sharma,0.6376,"[astronomy and stargazing, marvel movie nights, watching movies, watching sci-fi movies]"


**Sentence embeddings**: `all-MiniLM-L6-v2` embeds each interest phrase, so *cafe hopping* ≈ *drinking filter coffee*. `similar_interests` explains the semantic matches.

In [4]:
Recommender("embedding").fit(profiles).recommend(target, 5, cands)

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,user_id,name,score,shared_interests,similar_interests
0,20,Ananya Reddy,0.9180,"[astronomy and stargazing, marvel movie nights, scrolling social media, watching movie...","[star trek marathons ~ space documentaries (0.45), hanging out with friends ~ listenin..."
1,4,Arjun Bhat,0.9132,"[marvel movie nights, scrolling social media, star trek marathons, watching sci-fi mov...",[watching movies ~ reading science fiction novels (0.43)]
2,19,Vikram Kulkarni,0.9064,"[marvel movie nights, scrolling social media, star trek marathons, watching movies]","[watching sci-fi movies ~ reading science fiction novels (0.60), watching sci-fi movie..."
3,11,Ananya Menon,0.8936,"[astronomy and stargazing, scrolling social media, star trek marathons, watching movies]","[watching sci-fi movies ~ space documentaries (0.58), marvel movie nights ~ space docu..."
4,1,Isha Sharma,0.8842,"[astronomy and stargazing, marvel movie nights, watching movies, watching sci-fi movies]",[star trek marathons ~ space documentaries (0.45)]


**Hybrid**: α·TF-IDF + (1−α)·embedding. Exact shared rare tastes and near-synonyms both count.

In [5]:
Recommender("hybrid", alpha=0.5).fit(profiles).recommend(target, 5, cands)

,user_id,name,score,shared_interests,similar_interests
0,4,Arjun Bhat,0.8245,"[marvel movie nights, scrolling social media, star trek marathons, watching sci-fi mov...",[watching movies ~ reading science fiction novels (0.43)]
1,20,Ananya Reddy,0.7837,"[astronomy and stargazing, marvel movie nights, scrolling social media, watching movie...","[star trek marathons ~ space documentaries (0.45), hanging out with friends ~ listenin..."
2,16,Meera Rao,0.7758,"[hanging out with friends, marvel movie nights, star trek marathons, watching movies, ...",[]
3,1,Isha Sharma,0.7609,"[astronomy and stargazing, marvel movie nights, watching movies, watching sci-fi movies]",[star trek marathons ~ space documentaries (0.45)]
4,17,Varun Menon,0.7577,"[astronomy and stargazing, hanging out with friends, scrolling social media, watching ...",[watching movies ~ listening to music (0.48)]


### Hybrid + secondary signals
`score` = interest_score + 0.3·mutual friends (capped at 3, scaled 0–1) + 0.3·places in common (TF-IDF cosine) + 0.05·recency (exp(−days/30)).

In [6]:
Recommender("hybrid", alpha=0.5, weights=DEFAULT_WEIGHTS).fit(profiles, conns, signals).recommend(target, 5, cands)

,user_id,name,score,shared_interests,similar_interests,interest_score,mutual_friends,common_places,days_inactive
0,4,Arjun Bhat,1.1202,"[marvel movie nights, scrolling social media, star trek marathons, watching sci-fi mov...",[watching movies ~ reading science fiction novels (0.43)],0.8245,1,[Planetarium Night Shows],6
1,13,Karthik Verma,1.0759,"[hanging out with friends, marvel movie nights, scrolling social media, watching sci-f...","[watching movies ~ listening to music (0.48), star trek marathons ~ space documentarie...",0.7270,1,"[IronCore Gym, HSR]",3
2,23,Kabir Hegde,1.0676,"[hanging out with friends, scrolling social media, star trek marathons, watching sci-f...",[],0.7050,1,[Planetarium Night Shows],5
3,6,Sana Bhat,1.0363,"[astronomy and stargazing, marvel movie nights, scrolling social media]","[watching sci-fi movies ~ reading science fiction novels (0.60), watching movies ~ lis...",0.6776,1,"[IronCore Gym, HSR, Planetarium Night Shows]",43
4,5,Harsha Kulkarni,0.9299,"[astronomy and stargazing, hanging out with friends, watching movies, watching sci-fi ...",[],0.6671,1,[Planetarium Night Shows],91


### What the embedding model thinks is similar

In [7]:
from recsys.models import encode
phrases = ["watching sci-fi movies", "star trek marathons", "drinking filter coffee", "cafe hopping", "home baking", "listening to music"]
E = encode(phrases)
pd.DataFrame(E @ E.T, index=phrases, columns=phrases).round(2)

,watching sci-fi movies,star trek marathons,drinking filter coffee,cafe hopping,home baking,listening to music
watching sci-fi movies,1.00,0.33,0.11,0.07,0.00,0.35
star trek marathons,0.33,1.00,0.04,0.16,-0.02,0.02
drinking filter coffee,0.11,0.04,1.00,0.25,0.05,0.16
cafe hopping,0.07,0.16,0.25,1.00,0.28,0.23
home baking,0.00,-0.02,0.05,0.28,1.00,0.07
listening to music,0.35,0.02,0.16,0.23,0.07,1.00
